# Computer Vision Using Traditional Machine Learning

In this lab, we are going to learn about certain machine learning classifiers, and how we can extract features from images and train our model using these features.

## Outline 
* Learn how to use sklearn to create models.
* Learn how to extract features from an image using OpenCV.
* Understand the concepts of using features and their limitations.

## References 
* [SciKit User Guide](https://scikit-learn.org/stable/user_guide.html)
* [OpenCV - Feature Detection and Description](https://docs.opencv.org/4.5.5/db/d27/tutorial_py_table_of_contents_feature2d.html)


## 1: Importing libraries
The next code will contain the imports of various libraries, starting from NumPy, which is an essential library for our lab.
Notice that we will import extra libraries later.

In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mnist

#### Loading MNIST Database

In [2]:
(train_images, train_labels) = mnist.load_mnist("data/train-images-idx3-ubyte.gz", "data/train-labels-idx1-ubyte.gz")
(test_images, test_labels) = mnist.load_mnist("data/t10k-images-idx3-ubyte.gz", "data/t10k-labels-idx1-ubyte.gz")

## 2: Using k-NN With Row Pixels
The k-nearest neighbors algorithm will predict the class of an vector by calculating the distance between it and the whole training dataset, then using the parameter `k` it will predict the result, notice that the training phase will only store the data and labels, and the prediction will be a very slow process especially if we have a large training data.
<center>
<img src="./images/knn.png" width="200px" alt="k-NN Classifier" /> <br/>
k-NN Classifier <a href="https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm" target="_blank">[1]</a>
</center>

In [3]:
from sklearn.neighbors import KNeighborsClassifier

ModuleNotFoundError: No module named 'sklearn'

Notice that our images are `28x28` matrices, so we need to flatten them to a vector with `784` element in it.

In [ ]:
train_images_flat = train_images.reshape(60000, 28*28)
test_images_flat = test_images.reshape(10000, 28*28)

In [ ]:
model = KNeighborsClassifier(n_neighbors=3, n_jobs=8)
model.fit(train_images_flat,train_labels)

Now we will predict the first ten images:

In [ ]:
predicted = model.predict(test_images_flat[:10])
for i in range(10):
    print("Image "+ str(i) +" - Label: "+ str(test_labels[i]) +" - Prediction: "+ str(predicted[i]))

That is nice, we predicted the fisrt ten images correctly!

In [ ]:
# predicted_labels = model.predict(test_images_flat) # Notice this will take some time! (about 4 mins)
# Or you could use my results
predicted_labels = np.load('data/knn_predicted_labels.npy')

In [ ]:
correct_predictions = np.sum(predicted_labels == test_labels)
accuracy = correct_predictions / len(predicted_labels)
print("Model accuracy = "+str(accuracy*100) + "%")

That is promising, We used the row pixels intensity as learning features and we've got a very got accuracy (97%) compared to `Lab 3 Model` (82%).

Notice that our prediction is based on the intensity and position of the pixels.

## 3: Feature Detection and Description
You might play several Jigsaw games, but did you ever wonder how could you solve it, what techiness do you follow to solve this puzzle?

![Jigsaw Puzzle](./images/jigsaw.png)

We as humans find some kind of feature that helps us solve it faster, for example, the edge pieces are easier to solve, the flat color pieces are the hardest so we leave them to the end.

This is what we called features, it's something that describes the image, but not the image itself, so if we could extract certain features and use them in machine learning, our model will do better than feeding the whole image.

These features could be immune to intensity change, or scaling, this what makes a feature a good feature, it's describe something we need and is immune to noises.

### Histogram of Oriented Gradients (HOG) Feature

To get the histogram of oriented gradients according to its [paper](http://lear.inrialpes.fr/people/triggs/pubs/Dalal-cvpr05.pdf), you will need to do the following:
1. Resize the image
2. Get the X-Gradient and Y-Gradient
3. Get the magnitude and angle of the pixel gradient
4. Calculate the histogram of the angles with the magnitude as a weight over the cell (e.g. 8x8 pixels)
5. Normalize the values over blocks (e.g 2x2 cells)
6. Store the values of each block in a vector (each block will have a number of cells * histogram pins values).

**If you want to understand more you could read this [awesome blog](https://learnopencv.com/histogram-of-oriented-gradients/).**

In [ ]:
from skimage.feature import hog

image = cv2.imread('images/walking.jpg')
image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
resized_image = cv2.resize(image, (64,128))

# We will use skimage to get HOG features and visualization image (not useful for ML)
sk_fd, hog_image = hog(resized_image, cells_per_block=(2, 2), pixels_per_cell=(16, 16), 
                       visualize=True, orientations=9, block_norm='L2-Hys')

# The settings here are not an exact match to skimage
hog_cv = cv2.HOGDescriptor(_winSize=(64, 128), _blockSize=(32,32), 
                        _blockStride=(16,16), _cellSize=(16,16), _nbins=9)
cv_fd = hog_cv.compute(resized_image)

# The values for the two descriptors will differ
print ("skimage fd shape: "+str(sk_fd.shape))
print ("opencv  fd shape: "+str(cv_fd.shape))

In [ ]:
from skimage import exposure

sk_fd, hog_image = hog(image, cells_per_block=(2, 2), pixels_per_cell=(16, 16), 
                       visualize=True, orientations=9, block_norm='L2-Hys')

fig, axs = plt.subplots(1,2, figsize=(15,15))
axs[0].imshow(image, cmap=plt.cm.gray)
axs[0].set_title('Input')

# Just for visualization!
hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

axs[1].imshow(hog_image_rescaled, cmap=plt.cm.gray)
axs[1].set_title('Histogram of Oriented Gradients')
plt.show()

### Predict digits using HOG and SVC

In this section we will predict the MNIST digits using HOG features, and use these features as an input for [SVC model](https://scikit-learn.org/stable/modules/svm.html), the Support Vector Machines will take longer time in the training process but will predict the results way faster than k-NN.

<br />
<center>
<img alt="Feeding Features to a Machine Learning Model" src="./images/traditionalML.jpg" width="400px" /><br/>
    Feeding Features to a Machine Learning Model
</center>

In [ ]:
hog_mnist = cv2.HOGDescriptor(_winSize=(28, 28), _blockSize=(4,4), 
                        _blockStride=(4,4), _cellSize=(4,4), _nbins=9)

### Task 1: Compute the hog features for all training images.

In [ ]:
train_images_uint8 = train_images.astype('uint8')
train_images_hog = np.empty((train_images.shape[0], hog_mnist.getDescriptorSize()))
# Place your code here!
print(train_images_hog.shape)

In [ ]:
from sklearn import svm
clf = svm.LinearSVC()
clf.fit(train_images_hog, train_labels)

### Task 2: Compute the hog features for all test images.

In [ ]:
test_images_uint8 = test_images.astype('uint8')
test_images_hog = np.empty((test_images.shape[0], hog_mnist.getDescriptorSize()))
# Place your code here!

### Task 3: Predict the labels using `.predict`.

In [ ]:
%time
# Place your code here

You could use [metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics) from sklearn to evaluate your model.

In [ ]:
from sklearn import metrics
acc = metrics.accuracy_score(test_labels, predicted_labels)
print ("Accuracy: "+str(acc*100)+"%")

metrics.plot_confusion_matrix(clf, test_images_hog, test_labels)
plt.show()

## 4: Other Features Examples
In this section, we won't talk much about a certain feature, but we will try some features from OpenCV, you could alwayse return to the orignal paper and documentation to understand more.

### SIFT (Scale-Invariant Feature Transform)
Selecting corners is very sensitive to scaling (A hard edge could be smooth when we zoom in), so this is what SIFT solves basicly, please read more [here](https://docs.opencv.org/4.5.5/da/df5/tutorial_py_sift_intro.html).

In [ ]:
img = cv2.imread('images/walking.jpg')
gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp = sift.detect(gray,None)
img_keypoints = np.copy(img)
cv2.drawKeypoints(gray,kp,img_keypoints,flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(15,15))
plt.imshow(img_keypoints)
plt.show();

Notice that SIFT detected keypoints, which are the intersting areas in this image.
You can now use these keypoints and get a feature descripots using `sift.compute`.

In [ ]:
kp, des = sift.detectAndCompute(gray,None)
print(des.shape)

Notice that every keypoint has 128 features (the description of that point)

#### SURF (Speeded-Up Robust Features)
It's basicly a speeded-up version of SIFT, you can read more [here](https://docs.opencv.org/4.5.5/df/dd2/tutorial_py_surf_intro.html).


#### Features Matching

In this section, we will match keypoints featrues with another image keypoints features (which mean we will try to find a picture in a picture, we will use ORB (Oriented FAST and Rotated BRIEF) to create our keypoints and descriptors, and we will use BFMatcher (Brute-Force Matching) to match points.

You can read more [here](https://docs.opencv.org/4.5.5/dc/dc3/tutorial_py_matcher.html).


In [ ]:
img_on_screen = cv2.imread('images/my_pc.jpg',cv2.IMREAD_GRAYSCALE)

# Initiate ORB detector
orb = cv2.ORB_create()
# find the keypoints and descriptors with ORB
kp1, des1 = orb.detectAndCompute(img,None)
kp2, des2 = orb.detectAndCompute(img_on_screen,None)

# create BFMatcher object
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
# Match descriptors.
matches = bf.match(des1,des2)
# Sort them in the order of their distance.
matches = sorted(matches, key = lambda x:x.distance)

# Draw first 20 matches.
img3 = cv2.drawMatches(img,kp1,img_on_screen,kp2,matches[:20],None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(15,15))
plt.imshow(img3)
plt.show()

**You could also draw a box around what you find if you want to learn more [read this](https://docs.opencv.org/4.5.5/d1/de0/tutorial_py_feature_homography.html)**

### Haar Cascade Classifier
Haar feature-based cascade classifiers is an effective object detection method, you could learn more about it [here](https://docs.opencv.org/3.4/db/d28/tutorial_cascade_classifier.html), or you could watch [this video](https://www.youtube.com/watch?v=ZSqg-fZJ9tQ).

In this example, we will use already trained models, one for detecting faces, and the other for detecting eyes.
You can find the original example here: https://docs.opencv.org/3.4/db/d28/tutorial_cascade_classifier.html

**Please read the previous link, it has a lot of very creative and amazing ideas.**

In [ ]:
def detectAndDisplay(frame):
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_gray = cv2.equalizeHist(frame_gray)
    #-- Detect faces
    faces = face_cascade.detectMultiScale(frame_gray)
    for (x,y,w,h) in faces:
        frame = cv2.rectangle(frame, (x,y), (x+w, y+h), (255, 0, 255), 4)
        faceROI = frame_gray[y:y+h,x:x+w]
        #-- In each face, detect eyes
        eyes = eyes_cascade.detectMultiScale(faceROI)
        for (x2,y2,w2,h2) in eyes:
            eye_center = (x + x2 + w2//2, y + y2 + h2//2)
            radius = int(round((w2 + h2)*0.25))
            frame = cv2.circle(frame, eye_center, radius, (255, 0, 0 ), 4)
    cv2.imshow('Capture - Face detection', frame)

face_cascade = cv2.CascadeClassifier()
eyes_cascade = cv2.CascadeClassifier()

face_cascade.load("haar_data/haarcascade_frontalface_alt.xml")
eyes_cascade.load("haar_data/haarcascade_eye_tree_eyeglasses.xml")

cap = cv2.VideoCapture(0)
if not cap.isOpened:
    print('--(!)Error opening video capture')
else:
    while True:
        ret, frame = cap.read()
        detectAndDisplay(frame)
        if cv2.waitKey(10) == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

<center>
    <h4>End of Lab 5</h4>
</center>